# Deterministic ASR for all EXIOBASE sector and region pairs

This notebook prepares deterministic absolute sustainability ratio (ASR) results for every electricity aggregated EXIOBASE 3.10.2 ixi sector and region pair in 2022. It evaluates the five requested level 2 functional units with every default pyaesa allocation method.

The workflow uses retrospective MRIO data from 1995 through 2022, GWP100 IO LCA results, acquired rights reference years 1995 and 2022, and only the static `min_cc` carrying capacity bound. Native Phase A IO LCA, Phase B aSoCC and aCC, and Phase C ASR outputs are written as Parquet files. The last section reads these published pyaesa results and combines them into one CSV.


## 1. Workspace and configuration

`set_workspace(...)` initializes the pyaesa workspace folders and records the active workspace for later public function calls. The path below is the root that will contain the five project folders and `synthesis_fus`.


In [ ]:
from pathlib import Path

import pandas as pd

from pyaesa import (
    deterministic_asr,
    download_mrio,
    download_pop_gdp,
    process_mrio,
    process_pop_gdp,
    set_workspace,
)

# Replace this placeholder with the pyaesa workspace to use for the study.
WORKSPACE_ROOT = Path("/path/to/pyaesa_workspace")
SOURCE = "exiobase_3102_ixi"
MRIO_YEARS = range(1995, 2023)
STUDIED_YEAR = 2022
REFERENCE_YEARS = (1995, 2022)
LCIA_METHOD = "gwp100_lcia"
AGG_VERSION = "elec"
FU_CODES = ("L2.a.a", "L2.a.b", "L2.a.c", "L2.c.a", "L2.c.b")
IMPACT = "GWP_100"
CC_TYPE = "static"
CC_BOUND = "min_cc"
CC_TABLE_PATH = (
    WORKSPACE_ROOT / "data_raw" / "carrying_capacities" / "gwp100_lcia_cc_steady_state.csv"
)
SYNTHESIS_DIR = WORKSPACE_ROOT / "synthesis_fus"
SYNTHESIS_PATH = SYNTHESIS_DIR / "deterministic_asr_all_fus_2022.csv"

set_workspace(WORKSPACE_ROOT)

## 2. Download source data

Download the population and GDP inputs and the EXIOBASE years required by the analysis.


In [2]:
download_pop_gdp()
download_mrio(SOURCE, years=MRIO_YEARS)

## 3. Process source data

Process population and GDP data, then process EXIOBASE for 1995 through 2022 with GWP100. The packaged `elec` aggregation groups the EXIOBASE electricity sectors together.


In [3]:
process_pop_gdp()
process_mrio(
    source=SOURCE,
    years=MRIO_YEARS,
    lcia_method=LCIA_METHOD,
    agg_sec=True,
    agg_version=AGG_VERSION,
)

## 4. Compute deterministic ASR

Each functional unit uses its own project named with the exact FU code. Region and sector selectors and allocation method selectors are intentionally omitted so pyaesa expands every valid pair and applies its complete default method plan.

The request activates deterministic IO-LCA, selects the two acquired rights reference years, excludes `max_cc`, writes Parquet tables, and disables figures. No uncertainty source is activated.


In [4]:
asr_reports = {}
for fu_code in FU_CODES:
    asr_reports[fu_code] = deterministic_asr(
        project_name=fu_code,
        source=SOURCE,
        agg_sec=True,
        agg_version=AGG_VERSION,
        years=STUDIED_YEAR,
        fu_code=fu_code,
        lcia_method=LCIA_METHOD,
        base_asocc_args={"reference_years": list(REFERENCE_YEARS)},
        base_cc_args={"static": {"exclude_max_cc": True}},
        lca_args={"io_lca": {}},
        output_format="parquet",
        figures=False,
    )

## 5. Compile the published results

The synthesis uses the phase output folders returned by `deterministic_asr`. It reads the published pyaesa Parquet tables for Phase A IO LCA, Phase B aSoCC and aCC, and Phase C ASR. It also reads the selected static carrying capacity directly from the pyaesa input table and combines the 2022 values.


In [ ]:
PHASE_RESULTS = {
    "io_lca": ("deterministic_io_lca", "results", "lca_value"),
    "asocc": ("deterministic_asocc", "l2_vs_global", str(STUDIED_YEAR)),
    "acc": ("deterministic_acc", "results_l2_vs_global", str(STUDIED_YEAR)),
    "asr": ("deterministic_asr", "results_l2_vs_global", str(STUDIED_YEAR)),
}
IDENTITY_COLUMNS = (
    "l1_l2_method",
    "reference_year",
    "r_p",
    "r_c",
    "r_f",
    "s_p",
    "year",
)
FINAL_COLUMNS = (
    "fu_code",
    "source",
    "agg_version",
    "lcia_method",
    "impact",
    "impact_unit",
    "cc_type",
    "cc_bound",
    *IDENTITY_COLUMNS,
    "io_lca",
    "asocc",
    "cc",
    "acc",
    "asr",
)

In [6]:
def _read_phase_rows(
    report,
    *,
    function_name: str,
    result_dir_name: str,
    source_column: str,
    value_column: str,
) -> pd.DataFrame:
    """Read published pyaesa Parquet tables for one deterministic phase."""
    output_root = next(
        entry.output_root
        for branch in report.branches
        for entry in branch.phase_entries
        if entry.function == function_name
    )
    frames = []
    for path in sorted(output_root.rglob(f"{result_dir_name}/*.parquet")):
        frame = pd.read_parquet(path)
        frame = frame.rename(columns={source_column: value_column})
        frame["year"] = STUDIED_YEAR
        frames.append(frame)
    published = pd.concat(frames, ignore_index=True)
    identity_columns = [column for column in IDENTITY_COLUMNS if column in published]
    return published.loc[:, [*identity_columns, value_column]]

In [ ]:
def _merge_phase_value(
    left: pd.DataFrame,
    right: pd.DataFrame,
    value_column: str,
) -> pd.DataFrame:
    """Add one published phase value using shared pyaesa identity columns."""
    identity_columns = [column for column in IDENTITY_COLUMNS if column in left and column in right]
    return left.merge(
        right.loc[:, [*identity_columns, value_column]],
        on=identity_columns,
    )


def _compile_project_rows(fu_code: str, report) -> pd.DataFrame:
    """Combine published IO LCA, aSoCC, aCC, and ASR values for one project."""
    phase_rows = {
        value_column: _read_phase_rows(
            report,
            function_name=function_name,
            result_dir_name=result_dir_name,
            source_column=source_column,
            value_column=value_column,
        )
        for value_column, (function_name, result_dir_name, source_column) in PHASE_RESULTS.items()
    }
    combined = phase_rows["asr"]
    for value_column in ("io_lca", "asocc", "acc"):
        combined = _merge_phase_value(combined, phase_rows[value_column], value_column)
    combined.insert(0, "fu_code", fu_code)
    return combined

In [ ]:
cc_table = pd.read_csv(CC_TABLE_PATH)
cc_rows = cc_table.loc[cc_table["impact"].eq(IMPACT)]
if len(cc_rows) != 1:
    raise ValueError(f"Expected one carrying capacity row for {IMPACT}, found {len(cc_rows)}.")
cc_row = cc_rows.iloc[0]
cc_value = float(cc_row[CC_BOUND])
if pd.isna(cc_value) or cc_value <= 0:
    raise ValueError(f"The selected carrying capacity must be positive: {cc_value}.")

compiled = pd.concat(
    [_compile_project_rows(fu_code, asr_reports[fu_code]) for fu_code in FU_CODES],
    ignore_index=True,
)
compiled = compiled.assign(
    source=SOURCE,
    agg_version=AGG_VERSION,
    lcia_method=LCIA_METHOD,
    impact=IMPACT,
    impact_unit=str(cc_row["impact_unit"]),
    cc_type=CC_TYPE,
    cc_bound=CC_BOUND,
    cc=cc_value,
).loc[:, FINAL_COLUMNS]

duplicate_count = int(compiled.duplicated(subset=["fu_code", *IDENTITY_COLUMNS]).sum())
if duplicate_count:
    raise ValueError(f"The synthesis contains {duplicate_count} duplicate result rows.")
if compiled[["io_lca", "asocc", "acc"]].isna().any().any():
    raise ValueError("IO LCA, aSoCC, and aCC must be present in every result row.")

## 6. Write the compiled CSV

The CSV contains the functional unit, MRIO and LCIA configuration, allocation method, applicable reference year, native region axes, sector, and year. The calculation fields are `io_lca`, `asocc`, `cc`, `acc`, and `asr`. It is written to the `synthesis_fus` folder under the workspace configured with `set_workspace`.


In [ ]:
SYNTHESIS_PATH.parent.mkdir(parents=True, exist_ok=True)
compiled.to_csv(SYNTHESIS_PATH, index=False)

pd.Series(
    {
        "rows": len(compiled),
        "workspace_csv": str(SYNTHESIS_PATH),
    }
)